In [1]:
import numpy as np

from engin_core.gp import fit_gp, split_conformal_multiplier, smallest_calibration_set
from engin_core.simulator import simulate_unit

NOMINAL = 0.90
rng = np.random.default_rng(0)

# A 24-run DoE over the five process knobs, with realistic assay noise.
U = rng.random((24, 5))
y_true = simulate_unit(U)
y_obs = np.maximum(y_true + rng.normal(0, 0.05 * y_true + 0.4), 0.0)

print(f"{len(U)} runs, titer {y_obs.min():.1f}-{y_obs.max():.1f} g/L")

24 runs, titer 13.4-104.1 g/L


In [2]:
# 10 calibration points, not 8. Section 3 explains why that is not a free
# choice: at 90% the floor is 9, and below it the level is unavailable.
tr, ca = slice(0, 14), slice(14, 24)
gp = fit_gp(U[tr], y_obs[tr], seed=0)

mean_ca, sd_ca = gp.predict(U[ca], include_noise=True)
print(f"fitted on {tr.stop - tr.start}, calibrating on {ca.stop - ca.start}")

fitted on 14, calibrating on 10


In [3]:
import warnings

# Captured rather than left to stderr purely so this page's output is
# reproducible -- the raw warning carries a per-process temp path, and these
# docs are checked by re-executing them and diffing against the committed run.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    q = split_conformal_multiplier(y_obs[ca], mean_ca, sd_ca, level=NOMINAL)

gp.q90 = q                      # the wrapper carries it for downstream consumers
print(f"conformal multiplier q = {q:.2f}   (a Gaussian 90% would use 1.64)")
print(f"it also warned: {str(caught[0].message)[:96]} ...")

# The interval is mean +/- q*sd. Use the total predictive sd: you are predicting
# an assay result, and the assay has noise in it.
#
# Predict on designs the model has never seen -- not on the calibration set,
# which set q and would flatter the result.
Xs = rng.random((4, 5))
mean_s, sd_s = gp.predict(Xs, include_noise=True)
lo, hi = mean_s - q * sd_s, mean_s + q * sd_s
truth = simulate_unit(Xs)

for i in range(len(Xs)):
    covered = "yes" if lo[i] <= truth[i] <= hi[i] else "NO"
    print(f"new design {i}: {lo[i]:6.1f} - {hi[i]:6.1f} g/L   true {truth[i]:6.1f}   covered: {covered}")

conformal multiplier q = 2.56   (a Gaussian 90% would use 1.64)
it also warned: calibration set of 10 gives a 90% interval whose true coverage lies in roughly [0.74, 0.99] (cen ...
new design 0:   91.2 -  116.4 g/L   true  101.7   covered: yes
new design 1:  -11.2 -   38.2 g/L   true    5.3   covered: yes
new design 2:    3.6 -   50.8 g/L   true   24.5   covered: yes
new design 3:   20.1 -   43.6 g/L   true   34.6   covered: yes


In [4]:
for level in (0.80, 0.90, 0.95, 0.99):
    print(f"{level:.0%} interval needs at least {smallest_calibration_set(level):>3} calibration points")

80% interval needs at least   4 calibration points
90% interval needs at least   9 calibration points
95% interval needs at least  19 calibration points
99% interval needs at least  99 calibration points


In [5]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    split_conformal_multiplier(y_obs[ca], mean_ca, sd_ca, level=0.99)
    print(str(caught[0].message)[:200], "...")

calibration set of 10 is below the floor of 99 for a 99% split-conformal interval: ceil((n+1)*level) exceeds n, so the conformal quantile does not exist. Falling back to the largest calibration score, ...


In [6]:
from engin_core.gp import conformal_coverage_interval

for n_cal in (9, 20, 50, 406):
    lo_c, mean_c, hi_c = conformal_coverage_interval(n_cal, level=NOMINAL)
    print(f"n={n_cal:>3}: realised coverage lands in {lo_c:.2f}-{hi_c:.2f} (central 90%)")

n=  9: realised coverage lands in 0.72-0.99 (central 90%)
n= 20: realised coverage lands in 0.78-0.98 (central 90%)
n= 50: realised coverage lands in 0.83-0.96 (central 90%)
n=406: realised coverage lands in 0.88-0.92 (central 90%)


In [7]:
from engin_core.recommend import recommend_batch

X, mean, sd, ei = recommend_batch(gp, best_y=float(y_obs.max()), k=4, seed=0)
for i in range(len(X)):
    print(f"design {i}: predicted {mean[i]:5.1f} +/- {sd[i]:4.1f} g/L, EI {ei[i]:.3f}")

design 0: predicted 116.8 +/-  5.5 g/L, EI 12.630
design 1: predicted 115.1 +/-  6.0 g/L, EI 11.010
design 2: predicted 113.5 +/-  5.0 g/L, EI 9.387
design 3: predicted 111.3 +/-  4.3 g/L, EI 7.199
